In [ ]:
!pip install torch_snippets
from torch_snippets import transforms as T
from torch.nn import functional as F
from torchvision.models import vgg19  # usa uma VGG19 pré-treinada só como extrator de características, não pra classificar nada
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
preprocess = T.Compose([
    T.ToTensor(),
    # Media e std do ImageNet, precisa dos mesmo valores usados para o treino original da VGG
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Lambda(lambda x: x.mul_(255))
])
postprocess = T.Compose([
    T.Lambda(lambda x: x.mul__(1./255)),
    # desnormaliza pra voltar pra escala de imagem normal
    T.Normalize(mean=[-0.485/0.299, -0.456/0.224, -.0406/0.255], std=[1/0.229, 1/0.224, 1/0.225]),
])

In [ ]:
# a matriz de Gram captura estilo
class GramMatrix(nn.Module):  
  def forward(self, input):
    b,c,h,w = input.size()
    feat = input.view(b, c, h*w) # achata a dimensão espacial (h*w) mantendo os canais
    G = feat@feat.transpose(1,2) # multiplica o feature map por ele mesmo transposto e mede correlação entre canais
    G.div_(h*w) # normaliza pelo tamanho, senão imagens maiores dariam valores maiores artificialmente
    return G
class GramMSELoss(nn.Module):#compara a matriz de Gram gerada com a matriz de Gram alvo
  def forward(self, input, target):
    out = F.mse_loss(GramMatrix()(input), target)
    return(out)
class vgg19_modified(nn.Module):  # "abre" a VGG19 pra poder pegar a saída de camadas intermediárias específicas, não só a última
  def __init__(self):
    super().__init__()
    features = list(vgg19(pretrained = True).features)  # pega só a parte convolucional da VGG (sem as camadas densas finais)
    self.features = nn.ModuleList(features).eval()  # ModuleList guarda a lista de camadas mas deixa você controlar o forward manualmente
  def forward(self, x, layers=[]):
    order = np.argsort(layers)  # garante que os resultados saiam na mesma ordem em que "layers" foi passado, mesmo rodando em ordem crescente do índice real
    _results, results = [], []
    for ix, model in enumerate(self.features):
      x = model(x)
      if ix in layers: _results.append(x)  # vai guardando a saída só das camadas de interesse (estilo e conteúdo usam camadas diferentes)
    for o in order: results.append(_results[o])
    return results if layers is not [] else x  # obs: "is not []" quase nunca vale a pena (compara identidade de objeto, não conteúdo) - normalmente seria "if layers"</br>

In [5]:
vgg = vgg19_modified().to(device)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:11<00:00, 50.2MB/s]


In [ ]:
!wget https://easydrawinguides.com/wp-content/uploads/2016/10/how-to-draw-an-elephant-featured-image-1200-1024x822.png  # imagem de "conteúdo" (o que vai manter a estrutura)
!wget https://www.neh.gov/sites/default/files/2022-09/Fall_2022_web-images_Picasso_32.jpg  # imagem de "estilo" (de onde vai copiar a textura/pincelada)

In [12]:
!ls

Fall_2022_web-images_Picasso_32.jpg			  sample_data
how-to-draw-an-elephant-featured-image-1200-1024x822.png


In [ ]:
imgs = [Image.open(path).resize((512,512)).convert('RGB') for path in ['Fall_2022_web-images_Picasso_32.jpg', 'how-to-draw-an-elephant-featured-image-1200-1024x822.png']]
style_image, content_image = [preprocess(img).to(device)[None] for img in imgs]  # [None] adiciona a dimensão de batch em cada uma

In [ ]:
opt_img = content_image.data.clone()  # a imagem que vai ser otimizada começa como uma cópia da imagem de conteúdo
opt.img.required_grad = True  # bug: "opt.img" não existe (a variável se chama opt_img, com underscore) e o atributo certo seria "requires_grad", não "required_grad" -> essa linha não tem efeito nenhum, deveria ser opt_img.requires_grad = True

In [ ]:
style_layers = [0, 5, 10, 19, 28]  # índices das camadas da VGG usadas pra capturar ESTILO (várias camadas, de rasas a profundas)
content_layers = [21]               # e só uma camada mais profunda pra capturar CONTEÚDO/estrutura
loss_layers = style_layers + content_layers

In [ ]:
loss_fns = [GramMSELoss()] * len(style_layers) + [nn.MSELoss()] * len(content_layers)  # uma loss de estilo (Gram) pra cada camada de estilo, e MSE simples pra camada de conteúdo
loss_fns = [loss_fn.to(device) for loss_fn in loss_fns]

In [ ]:
style_weights = [1000/n**2 for n in [64,128,256,512,512]]  # pesos calculados pra compensar o número de canais de cada camada (senão camadas com mais canais dominariam a loss)
content_weights = [1]
weights = style_weights + content_weights

In [ ]:
style_targets = [GramMatrix()(A).detach() for A in vgg(style_image, style_layers)]  # calcula a matriz de Gram "alvo" a partir da imagem de estilo, uma vez só (detach pra não gerar gradiente aqui)
content_targets = [A.detach() for A in vgg(content_image, content_layers)]           # e guarda a ativação "alvo" da imagem de conteúdo
targets = style_targets + content_targets

In [ ]:
max_iters = 500
optimizer = optim.LBFGS([opt_img])  # LBFGS é um otimizador diferente do Adam/SGD, converge bem rápido pra esse tipo de otimização (poucos parâmetros: só os pixels da imagem)
log = Report(max_iters)

In [ ]:
iters = 0
while iters < max_iters:
  def closure():  # LBFGS precisa de uma "closure": uma função que recalcula a loss e o gradiente toda vez que é chamada (ele pode chamar várias vezes por passo)
    global iters
    iters  += 1
    optimizer.zero_grad()
    out = vgg(opt_img, loss_layers)
    layer_losses = [weights[a] * loss_fns[a](A, targets[a]) for a,A in enumerate(out)]  # soma ponderada da loss de cada camada (estilo + conteúdo)
    loss = sum(layer_losses)
    loss.backward()
    log.record(pos=iters, loss=loss, end='\r')
    return loss
  optimizer.step(closure)  # repara: quem sofre a atualização de gradiente aqui é a IMAGEM (opt_img), não os pesos de nenhuma rede - a VGG fica congelada o tempo todo

In [ ]:
log.plot(log=True)

In [ ]:
with torch.no_grad():
  out_img = postprocess(opt_img[0]).permute(1,2,0)  # desfaz a normalização pra voltar pra escala de imagem visualizável
show(out_img)